# 1.Importacion de librerías requeridas

In [0]:
%python
#Python para traer los datos desde la API

import requests   #Para poder traer la información de la Api. Permite conectarse a servicios externos mediante http
import pandas as pd #libreria python con funciones para manipulación de tablas 

from pandas import json_normalize #herramienta en python para convertir JSON anidado en filas y columnas


from pyspark.sql import functions as f #librería sirve paraconectar Python con Spark(motos de procesamiento de Databricks ) y aplicar funciones matemáticas y de texto a gran escala

from pyspark.sql.functions import current_timestamp


# 2.Definición de parámetros de extracción

Primero, necesitamos tres/cuatro  elementos:
- Token: Si la Api es privada
- API URL: dirección del servicio que queremos consultar.
- Headers: información adicional que enviamos en la solicitud.
- Parameters: filtros o condiciones que podemos enviar para controlar los resultados.

In [0]:
%python

#a. Definición de vaaribles necesarias para extracción 

# Token
api_token= "no requerido" 
 
#EndPoint
api_url= "https://api.github.com/users"  

#Headers
api_headers=  {  "Accept": "application/vnd.github+json",
                 "X-GitHub-Api-Version": "2026-03-10"}    

# Query parameters
api_params = {
    "per_page": 10,
    "since": 0
}



# 3. Conexión y descarga de datos desde la API

OBJETIVO:
Enviar una solicitud HTTP GET a GitHub, recibir la respuesta del servidor y almacenar los datos en una variable de Python.

In [0]:
%python

# Solicitud de conexión a la Api

response= requests.get(
    api_url,
    headers=api_headers,
    params=api_params,
    timeout= 30)  ##timeout: establece un tiempo maximo de respuesta de 30 segundos 

#validación de la respuesta

print("status code", response.status_code)
print("URL consultada:", response.url)

#obtener 200 indica conexión exitosa

status code 200
URL consultada: https://api.github.com/users?per_page=10&since=0


In [0]:
%python
# c. validación de la respuesta original. Response no contiene directamente una tabla. Es un objeto JSON que almacena información sobre la respuesta HTTP:

print(response.text[:1000])

[{"login":"mojombo","id":1,"node_id":"MDQ6VXNlcjE=","avatar_url":"https://avatars.githubusercontent.com/u/1?v=4","gravatar_id":"","url":"https://api.github.com/users/mojombo","html_url":"https://github.com/mojombo","followers_url":"https://api.github.com/users/mojombo/followers","following_url":"https://api.github.com/users/mojombo/following{/other_user}","gists_url":"https://api.github.com/users/mojombo/gists{/gist_id}","starred_url":"https://api.github.com/users/mojombo/starred{/owner}{/repo}","subscriptions_url":"https://api.github.com/users/mojombo/subscriptions","organizations_url":"https://api.github.com/users/mojombo/orgs","repos_url":"https://api.github.com/users/mojombo/repos","events_url":"https://api.github.com/users/mojombo/events{/privacy}","received_events_url":"https://api.github.com/users/mojombo/received_events","type":"User","user_view_type":"public","site_admin":false},{"login":"defunkt","id":2,"node_id":"MDQ6VXNlcjI=","avatar_url":"https://avatars.githubusercontent.

# 4. Conversión de la estructura Jeison a  una estrucura de Python

OBJETIVO
Transformar la respuesta recibida en una estructura de Python que podamos manipular.

In [0]:
%python
#a. Conversión de respeusta Json a objeto Python 

data=response.json()

#visualización de primeros registros y verificación de datos

print("tipo de dato:", type(data))
print("cantidad de registros:",  len(data))
print( data[:1])

tipo de dato: <class 'list'>
cantidad de registros: 10
[{'login': 'mojombo', 'id': 1, 'node_id': 'MDQ6VXNlcjE=', 'avatar_url': 'https://avatars.githubusercontent.com/u/1?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/mojombo', 'html_url': 'https://github.com/mojombo', 'followers_url': 'https://api.github.com/users/mojombo/followers', 'following_url': 'https://api.github.com/users/mojombo/following{/other_user}', 'gists_url': 'https://api.github.com/users/mojombo/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/mojombo/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/mojombo/subscriptions', 'organizations_url': 'https://api.github.com/users/mojombo/orgs', 'repos_url': 'https://api.github.com/users/mojombo/repos', 'events_url': 'https://api.github.com/users/mojombo/events{/privacy}', 'received_events_url': 'https://api.github.com/users/mojombo/received_events', 'type': 'User', 'user_view_type': 'public', 'site_admin': False}]


# 4. Normalización de datos JSON

OBJETIVO

Transformar los registros recibidos desde GitHub en una tabla con filas y columnas (dataframe), para facilitar su análisis y posterior almacenamiento

In [0]:
%python

#a. Normalización
df_users= json_normalize(data)

#b. validación del resultado

print("cantidad de registros:", len(df_users))
print("cantidad de filas", df_users.shape[0])
print("cantidad de columnas", df_users.shape[1])

#c visualización de resultados

display(df_users.head(2))

cantidad de registros: 10
cantidad de filas 10
cantidad de columnas 19


login,id,node_id,avatar_url,gravatar_id,url,html_url,followers_url,following_url,gists_url,starred_url,subscriptions_url,organizations_url,repos_url,events_url,received_events_url,type,user_view_type,site_admin
mojombo,1,MDQ6VXNlcjE=,https://avatars.githubusercontent.com/u/1?v=4,,https://api.github.com/users/mojombo,https://github.com/mojombo,https://api.github.com/users/mojombo/followers,https://api.github.com/users/mojombo/following{/other_user},https://api.github.com/users/mojombo/gists{/gist_id},https://api.github.com/users/mojombo/starred{/owner}{/repo},https://api.github.com/users/mojombo/subscriptions,https://api.github.com/users/mojombo/orgs,https://api.github.com/users/mojombo/repos,https://api.github.com/users/mojombo/events{/privacy},https://api.github.com/users/mojombo/received_events,User,public,false
defunkt,2,MDQ6VXNlcjI=,https://avatars.githubusercontent.com/u/2?v=4,,https://api.github.com/users/defunkt,https://github.com/defunkt,https://api.github.com/users/defunkt/followers,https://api.github.com/users/defunkt/following{/other_user},https://api.github.com/users/defunkt/gists{/gist_id},https://api.github.com/users/defunkt/starred{/owner}{/repo},https://api.github.com/users/defunkt/subscriptions,https://api.github.com/users/defunkt/orgs,https://api.github.com/users/defunkt/repos,https://api.github.com/users/defunkt/events{/privacy},https://api.github.com/users/defunkt/received_events,User,public,false


# 5. Creación de la estructura de almacenamiento - Catalog, schema 

OBJETIVO

Crear un espacio en Databricks donde podamos almacenar los datos que extraemos de GitHub.

In [0]:
--a) creación de catalogos y esquemas 

CREATE CATALOG IF NOT EXISTS github_data;
CREATE SCHEMA IF NOT EXISTS  github_data.bronze;
CREATE SCHEMA IF NOT EXISTS  github_data.silver;
CREATE SCHEMA IF NOT EXISTS  github_data.gold;

# 6. Almacenamiento de datos en Bronze

OBJETIVO

Tomar la información que tenemos en Python (data) y almacenarla de forma persistente en una tabla Delta dentro de nuestro esquema Bronze.

## Conversión de pandas a spark

In [0]:
%python

#a) converción de Pandas DataFrame a spark Dataframe (para poder trabajar con el motor de spark)

from pyspark.sql import functions as F

df_users_spark = spark.createDataFrame(df_users)

df_users_spark = df_users_spark.withColumn(
    "ingestion_timestamp",
    F.date_format(
        F.from_utc_timestamp(
            F.current_timestamp(),
            "America/Bogota"
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
)
#b.Verificar datos
print("cantidad de registros", df_users_spark.count())
display(df_users_spark.limit(2))


cantidad de registros 10


login,id,node_id,avatar_url,gravatar_id,url,html_url,followers_url,following_url,gists_url,starred_url,subscriptions_url,organizations_url,repos_url,events_url,received_events_url,type,user_view_type,site_admin,ingestion_timestamp
mojombo,1,MDQ6VXNlcjE=,https://avatars.githubusercontent.com/u/1?v=4,,https://api.github.com/users/mojombo,https://github.com/mojombo,https://api.github.com/users/mojombo/followers,https://api.github.com/users/mojombo/following{/other_user},https://api.github.com/users/mojombo/gists{/gist_id},https://api.github.com/users/mojombo/starred{/owner}{/repo},https://api.github.com/users/mojombo/subscriptions,https://api.github.com/users/mojombo/orgs,https://api.github.com/users/mojombo/repos,https://api.github.com/users/mojombo/events{/privacy},https://api.github.com/users/mojombo/received_events,User,public,false,2026-11-01 22:23:32
defunkt,2,MDQ6VXNlcjI=,https://avatars.githubusercontent.com/u/2?v=4,,https://api.github.com/users/defunkt,https://github.com/defunkt,https://api.github.com/users/defunkt/followers,https://api.github.com/users/defunkt/following{/other_user},https://api.github.com/users/defunkt/gists{/gist_id},https://api.github.com/users/defunkt/starred{/owner}{/repo},https://api.github.com/users/defunkt/subscriptions,https://api.github.com/users/defunkt/orgs,https://api.github.com/users/defunkt/repos,https://api.github.com/users/defunkt/events{/privacy},https://api.github.com/users/defunkt/received_events,User,public,false,2026-11-01 22:23:32


## Alimentación de la tabla 

In [0]:
%python

#Crear tabla destino

table_name="github_data.bronze.github_users_raw"

# Llenado de la tabla
df_users_spark.write\
.format("delta")\
.mode("overwrite")\
.option("overwriteSchema", "true")\
.saveAsTable(table_name)


## EDA  datos crudos

In [0]:
--Cantidad de registros

SELECT 
    COUNT(*) as  total_records
FROM github_data.bronze.github_users_raw

total_records
10


In [0]:
--Duplicados


SELECT
    id, 
    count(*) as total_records
FROM github_data.bronze.github_users_raw
GROUP BY id
HAVING count(*) >1


id,total_records


In [0]:
with duplicados as (

        SELECT
            count(*) as total_records
        FROM github_data.bronze.github_users_raw
        GROUP BY id
        HAVING count(*) >1
)

select
    (SELECT COUNT(*) FROM github_data.bronze.github_users_raw) as total_registros,
    COUNT(d.total_records) as duplicados
from duplicados as d

total_registros,duplicados
10,0


In [0]:
-- C. Registros con identificadores o usuarios nulos

SELECT
    count(*) as incomplete_Records
FROM github_data.bronze.github_users_raw
WHERE id is NULL 
    OR login is NULL


incomplete_Records
0
